# Chapter 4: RL for Language Models — From Grid Worlds to Text

> **Book:** Agentic AI Engineering  
> **Part:** 2 — RL Methods  
> **System:** Guardian Angel System (GAS) — Multi-agent diabetic patient supervision

In [ ]:
# Install dependencies
# pip install transformers datasets trl torch accelerate

In [ ]:
import os
TEST_MODE = os.getenv("NOTEBOOK_TEST_MODE", "0") == "1"
print(f"TEST_MODE: {TEST_MODE}")

---
## 🎯 1. Chapter Goal

By the end of this notebook you will:
- Understand the **SFT pipeline**: data → format → train → evaluate
- Create a **medical Q&A dataset** for the GAS Advisor Agent
- Format data with **chat templates**
- Run **SFT training** with TRL's SFTTrainer
- Understand **why SFT alone isn't enough** for safety-critical applications

---
## 📖 2. Theory Recap

### 2.1 SFT Pipeline

```
Pre-trained LLM (next-token prediction on internet text)
         ↓
Supervised Fine-Tuning (SFT)
  - Dataset: (prompt, ideal_response) pairs
  - Loss: cross-entropy on response tokens only
  - Result: instruction-following model
         ↓
RLHF / DPO / GRPO (alignment)
  - Dataset: (prompt, chosen, rejected) triples
  - Result: aligned, safe model
```

### 2.2 Instruction Following

SFT teaches the model to:
1. **Follow the format**: respond in the expected structure
2. **Stay on topic**: answer the question asked
3. **Use domain knowledge**: apply medical knowledge correctly

### 2.3 Chat Templates

Modern models use structured templates:
```
<|system|>You are a medical advisor...</s>
<|user|>Patient glucose is 280 mg/dL...</s>
<|assistant|>Based on the glucose reading...</s>
```

### 2.4 Why SFT Alone Isn't Enough

| Problem | Example | Solution |
|---------|---------|----------|
| Hallucination | Invents drug interactions | RLHF reward model |
| Overconfidence | "Definitely take 4 units" | Calibration training |
| Unsafe advice | "Skip your insulin today" | Safety critic (DPO/PPO) |
| Sycophancy | Agrees with wrong patient claims | Preference optimization |

---
## 🔨 3. Build It

### 3.1 Medical Q&A Dataset

In [ ]:
import json

# Medical Q&A dataset for GAS Advisor Agent SFT
# 20 high-quality (prompt, response) pairs
medical_qa_dataset = [
    {
        "prompt": "Patient glucose is 280 mg/dL, 2 hours after dinner. Insulin on board: 1.5 units. What should I do?",
        "response": "With glucose at 280 mg/dL post-meal and 1.5 units IOB, I recommend: (1) Wait 30-60 minutes before correcting — the IOB may still be working. (2) If glucose hasn't dropped in 1 hour, apply correction dose using your correction factor. (3) Check for ketones if glucose exceeds 300 mg/dL. (4) Avoid stacking insulin — the existing IOB reduces your correction need. Contact your care team if glucose remains >300 mg/dL for 2+ hours."
    },
    {
        "prompt": "CGM alert: glucose dropping rapidly from 180 to 95 mg/dL in 30 minutes. Trend arrow pointing down. What action?",
        "response": "A rapid drop of 85 mg/dL in 30 minutes requires immediate attention. Action steps: (1) Consume 15-20g fast-acting carbohydrates NOW (glucose tablets, juice, or regular soda). (2) Do NOT exercise or take additional insulin. (3) Recheck glucose in 15 minutes. (4) If still dropping or below 70, repeat carb intake. (5) Identify cause: recent insulin dose, missed meal, or unusual activity. The downward trend arrow indicates this will continue — act proactively before reaching hypoglycemia."
    },
    {
        "prompt": "Patient has Type 1 diabetes, HbA1c 9.2%, average glucose 245 mg/dL over past 30 days. What does this indicate?",
        "response": "HbA1c of 9.2% with average glucose of 245 mg/dL indicates poorly controlled Type 1 diabetes. Clinical interpretation: (1) HbA1c 9.2% corresponds to estimated average glucose of ~218 mg/dL — the 245 mg/dL average suggests significant hyperglycemia. (2) Target for most T1D patients: HbA1c <7.0% (average glucose ~154 mg/dL). (3) This level increases risk of: retinopathy, nephropathy, neuropathy, and cardiovascular disease. Recommended actions: Review basal/bolus insulin regimen, assess carb counting accuracy, consider CGM if not already using, and schedule endocrinology consultation."
    },
    {
        "prompt": "Glucose is 58 mg/dL. Patient feels shaky and sweaty. What is the immediate treatment?",
        "response": "Glucose of 58 mg/dL with symptoms indicates hypoglycemia requiring immediate treatment. Follow the 15-15 Rule: (1) IMMEDIATELY consume 15g fast-acting carbohydrates: 4 glucose tablets, 4 oz (120mL) fruit juice, 4 oz regular soda, or 1 tablespoon honey. (2) Wait 15 minutes, then recheck glucose. (3) If still below 70 mg/dL, repeat 15g carbs. (4) Once glucose is above 70 mg/dL, eat a small snack with protein and complex carbs. (5) Do NOT give insulin. Do NOT exercise. If patient is unconscious or cannot swallow, use glucagon emergency kit and call 911."
    },
    {
        "prompt": "Patient is about to eat a meal with approximately 60g of carbohydrates. Insulin-to-carb ratio is 1:15. Current glucose is 120 mg/dL. How much insulin?",
        "response": "Calculating mealtime insulin: (1) Carb dose: 60g ÷ 15 (I:C ratio) = 4 units. (2) Correction dose: Current glucose (120) is within target range (80-140 mg/dL), so no correction needed. (3) Total bolus: 4 units. Timing: Inject 15-20 minutes before eating for rapid-acting insulin (lispro, aspart, glulisine) to match glucose peak. Note: Adjust for meal composition — high-fat meals (pizza, pasta) may require extended or dual-wave bolus. Monitor glucose 2 hours post-meal; target <180 mg/dL."
    },
    {
        "prompt": "What is the difference between basal and bolus insulin?",
        "response": "Basal and bolus insulin serve different physiological roles: BASAL insulin: Provides background insulin coverage 24/7. Mimics the pancreas's constant low-level insulin secretion. Examples: glargine (Lantus), detemir (Levemir), degludec (Tresiba). Typically 40-50% of total daily dose. Keeps glucose stable between meals and overnight. BOLUS insulin: Covers carbohydrates eaten at meals (carb bolus) and corrects high glucose (correction bolus). Examples: lispro (Humalog), aspart (NovoLog), glulisine (Apidra). Fast-acting: peaks in 1-2 hours, duration 3-5 hours. Typically 50-60% of total daily dose. For optimal control, both types must be properly balanced."
    },
    {
        "prompt": "Patient glucose has been consistently high (200-280 mg/dL) every morning before breakfast for the past week. What could cause this?",
        "response": "Consistently high fasting glucose (200-280 mg/dL) has three main causes: (1) DAWN PHENOMENON (most common): Natural cortisol and growth hormone surge between 2-8 AM raises glucose. Solution: Increase basal insulin dose or adjust pump basal rate for early morning hours. (2) SOMOGYI EFFECT (rebound hyperglycemia): Overnight hypoglycemia triggers counter-regulatory hormones. Check 2-3 AM glucose — if low, reduce evening basal. (3) INSUFFICIENT BASAL INSULIN: Basal dose simply too low. Solution: Increase long-acting insulin by 10-20% (consult physician). Diagnostic step: Check glucose at 2-3 AM for 3 nights to distinguish between these causes."
    },
    {
        "prompt": "What are the symptoms of diabetic ketoacidosis (DKA) and when should a patient go to the emergency room?",
        "response": "Diabetic Ketoacidosis (DKA) is a life-threatening emergency. Symptoms: Early: Excessive thirst, frequent urination, nausea, abdominal pain, fatigue. Progressive: Vomiting, fruity-smelling breath (acetone), deep/rapid breathing (Kussmaul breathing), confusion. Severe: Loss of consciousness. Go to the ER IMMEDIATELY if: Glucose >300 mg/dL AND moderate/large ketones in urine or blood. Vomiting and unable to keep fluids down. Confusion or altered consciousness. Rapid breathing or fruity breath odor. Cannot reach healthcare provider within 1 hour. DKA can develop within hours in Type 1 diabetes. Do NOT wait to see if it improves. Call 911 if patient is unconscious."
    },
    {
        "prompt": "Patient is exercising for 45 minutes. Glucose was 140 mg/dL before exercise. How should insulin be adjusted?",
        "response": "Exercise management for Type 1 diabetes requires careful planning. Pre-exercise glucose of 140 mg/dL: (1) This is a good starting glucose for exercise (target: 126-180 mg/dL). (2) For aerobic exercise (running, cycling): Reduce bolus insulin by 25-75% for the meal before exercise. Consider 15-20g carbs if glucose drops below 126 mg/dL during exercise. (3) For pump users: Reduce basal rate by 50-80% starting 60-90 minutes before exercise. (4) Post-exercise: Glucose may drop for 6-12 hours (delayed hypoglycemia risk). Reduce overnight basal by 20%. Keep fast-acting carbs accessible during and after exercise. Recheck glucose every 30 minutes during exercise."
    },
    {
        "prompt": "What is continuous glucose monitoring (CGM) and how does it differ from fingerstick testing?",
        "response": "Continuous Glucose Monitoring (CGM) vs Fingerstick: CGM: Measures interstitial fluid glucose every 1-5 minutes via a small sensor inserted under the skin. Provides trend arrows (rising/falling rate). Alerts for high/low glucose. 10-14 day sensor life. 5-15 minute lag behind blood glucose. Examples: Dexcom G7, FreeStyle Libre 3, Medtronic Guardian. FINGERSTICK: Measures blood glucose directly. More accurate (no lag). Required for calibration and treatment decisions when CGM reads unusually high/low. Immediate single reading. KEY ADVANTAGE of CGM: Trend data. A glucose of 120 mg/dL with a rapidly falling arrow (↓↓) requires different action than 120 mg/dL with a stable arrow (→). CGM enables proactive management rather than reactive."
    },
    {
        "prompt": "Patient glucose is 45 mg/dL and they are unconscious. What should a bystander do?",
        "response": "EMERGENCY: Glucose of 45 mg/dL with unconsciousness is severe hypoglycemia requiring immediate action. DO NOT give food or liquid by mouth — aspiration risk. IMMEDIATE ACTIONS: (1) Call 911 immediately. (2) If glucagon kit available: Administer glucagon — either inject (1mg IM) or use nasal glucagon (Baqsimi). (3) Position patient on their side (recovery position) to prevent choking. (4) Stay with patient until emergency services arrive. (5) If no glucagon available: Apply glucose gel or honey to inside of cheek (small amount, do not force). AFTER RECOVERY: Patient will need medical evaluation. Do not leave alone. Identify and address cause. This is a medical emergency — do not delay calling 911."
    },
    {
        "prompt": "How should insulin be stored?",
        "response": "Proper insulin storage is critical for maintaining effectiveness. UNOPENED insulin: Store in refrigerator (36-46°F / 2-8°C). Do not freeze — freezing destroys insulin. Keep away from direct sunlight and heat. Check expiration date. OPENED/IN-USE insulin: Most insulins can be kept at room temperature (below 77-86°F / 25-30°C) for 28-30 days. Keep away from direct sunlight and heat sources. Do not store in car (temperature extremes). INSULIN PUMP: Reservoir insulin: Replace every 2-3 days. Discard if cloudy (clear insulins should be clear). SIGNS OF DAMAGED INSULIN: Cloudy appearance in normally clear insulin. Clumps or particles. Color change. Frosted appearance. Reduced effectiveness (unexplained high glucose). When in doubt, use a new vial."
    },
    {
        "prompt": "Patient reports glucose of 320 mg/dL with moderate ketones. They have not taken insulin in 12 hours. What is the protocol?",
        "response": "Glucose 320 mg/dL + moderate ketones + 12 hours without insulin = HIGH RISK for DKA. This requires urgent medical attention. IMMEDIATE STEPS: (1) Call healthcare provider or go to ER — do not manage this at home without guidance. (2) If provider advises home management: Administer correction dose of rapid-acting insulin (calculate based on correction factor). Drink water aggressively (8oz every hour). Recheck glucose and ketones every 2 hours. (3) GO TO ER IMMEDIATELY if: Vomiting occurs. Ketones increase to large. Glucose does not decrease after 2 hours. Confusion, difficulty breathing, or abdominal pain develops. DO NOT exercise — this worsens DKA. DO NOT skip further insulin doses."
    },
    {
        "prompt": "What is the target HbA1c for a Type 1 diabetic adult?",
        "response": "HbA1c targets for Type 1 diabetes (per ADA 2024 guidelines): GENERAL ADULT TARGET: <7.0% (estimated average glucose ~154 mg/dL). INDIVIDUALIZED TARGETS: More stringent (<6.5%): Patients with low hypoglycemia risk, short disease duration, long life expectancy, no cardiovascular disease. Less stringent (<8.0%): History of severe hypoglycemia, limited life expectancy, advanced complications, hypoglycemia unawareness, poor adherence. PREGNANCY: <6.0-6.5% (if achievable without hypoglycemia). TIME-IN-RANGE (TIR) is increasingly preferred: Target >70% of readings in 70-180 mg/dL. <4% below 70 mg/dL. <25% above 180 mg/dL. HbA1c should be checked every 3 months if not at goal, every 6 months if stable."
    },
    {
        "prompt": "Patient is sick with a cold and glucose is running high (200-300 mg/dL). Should they take more insulin?",
        "response": "Sick day management for Type 1 diabetes: Illness causes stress hormones that raise glucose — insulin needs typically INCREASE by 20-50% during illness. SICK DAY RULES: (1) NEVER stop taking insulin, even if not eating. (2) Check glucose every 2-4 hours. (3) Check ketones if glucose >240 mg/dL. (4) Stay hydrated: 8oz sugar-free fluids every hour. (5) If unable to eat: Replace carbs with liquids (broth, juice, regular ginger ale in small amounts). (6) Increase correction doses as needed per sick day protocol. CALL DOCTOR if: Glucose >300 mg/dL for 2+ hours. Moderate/large ketones. Vomiting for >2 hours. Confusion or difficulty breathing. Fever >101°F (38.3°C) for >24 hours. Have a sick day plan prepared with your care team before illness occurs."
    },
    {
        "prompt": "What is insulin resistance and how does it affect Type 2 diabetes management?",
        "response": "Insulin resistance is the hallmark of Type 2 diabetes. DEFINITION: Cells (muscle, liver, fat) respond poorly to insulin signals, requiring more insulin to achieve the same glucose-lowering effect. CAUSES: Obesity (especially visceral fat), physical inactivity, genetics, aging, certain medications (steroids, antipsychotics). PROGRESSION: Pancreas compensates by producing more insulin → eventually pancreatic beta cells exhaust → insulin deficiency develops. MANAGEMENT IMPLICATIONS: (1) Lifestyle: Weight loss (5-10% reduces insulin resistance significantly), exercise (150 min/week aerobic + resistance training). (2) Medications: Metformin (first-line), GLP-1 agonists (semaglutide, liraglutide), SGLT-2 inhibitors (empagliflozin, dapagliflozin), thiazolidinediones. (3) Insulin therapy: May be needed as disease progresses. Higher doses required than in Type 1."
    },
    {
        "prompt": "Patient glucose is 95 mg/dL and they are about to go to sleep. Should they have a bedtime snack?",
        "response": "Glucose of 95 mg/dL at bedtime requires careful consideration. ASSESSMENT: 95 mg/dL is at the lower end of normal range. Overnight, glucose can drop further (especially with active insulin or recent exercise). RECOMMENDATION: YES, a bedtime snack is advisable. Suggested snack: 15-20g complex carbohydrates + protein. Examples: 1 slice whole grain bread with peanut butter, small bowl of oatmeal, cheese and crackers, Greek yogurt. WHEN A SNACK IS ESSENTIAL: Glucose <100 mg/dL at bedtime. Recent exercise. Active insulin on board. History of overnight hypoglycemia. WHEN TO SKIP: Glucose >150 mg/dL. No active insulin. No recent exercise. Set CGM alert for <70 mg/dL overnight. Target bedtime glucose: 100-140 mg/dL for most patients."
    },
    {
        "prompt": "What is hypoglycemia unawareness and why is it dangerous?",
        "response": "Hypoglycemia unawareness is a serious complication of long-standing diabetes. DEFINITION: Loss of the normal warning symptoms (shakiness, sweating, hunger) that alert a person to low blood glucose. CAUSE: Repeated hypoglycemic episodes blunt the counter-regulatory hormone response (epinephrine, glucagon). The brain adapts to lower glucose levels. PREVALENCE: Affects 20-40% of people with Type 1 diabetes after 10+ years. WHY DANGEROUS: Person may have glucose of 40-50 mg/dL with NO symptoms. First sign may be confusion, seizure, or loss of consciousness. Cannot self-treat. MANAGEMENT: (1) Strict avoidance of hypoglycemia for 2-3 weeks can partially restore awareness. (2) Raise glucose targets temporarily. (3) CGM with low alerts is essential. (4) Educate family/caregivers on glucagon use. (5) Consider islet transplantation in severe cases. (6) Structured hypoglycemia awareness training (BGAT, HARPdoc programs)."
    },
    {
        "prompt": "Patient has been using an insulin pump for 3 days and glucose is consistently high. What should be checked?",
        "response": "Consistently high glucose 3 days into pump therapy requires systematic troubleshooting. CHECK IN ORDER: (1) INFUSION SITE: Is the cannula kinked or dislodged? Is there redness, swelling, or leakage? Change site immediately if any issues. Sites should be rotated every 2-3 days. (2) INSULIN: Is the reservoir empty? Is the insulin expired or damaged (cloudy)? Was insulin stored properly? (3) TUBING: Air bubbles in tubing? Tubing kinked or disconnected? Prime tubing after site change? (4) PUMP SETTINGS: Correct basal rates programmed? Correct I:C ratio and correction factor? Time and date correct? (5) TECHNIQUE: Proper site preparation (clean, dry skin)? Correct insertion angle? (6) INSULIN POTENCY: Pump insulin degrades faster than injected — replace reservoir every 2-3 days. If glucose remains >300 mg/dL after troubleshooting, inject correction dose via syringe (bypasses pump) and contact pump support line."
    },
    {
        "prompt": "What are the long-term complications of poorly controlled diabetes?",
        "response": "Long-term complications of poorly controlled diabetes affect multiple organ systems. MICROVASCULAR (small blood vessels): Retinopathy: Damage to retinal blood vessels → vision loss, blindness. Leading cause of blindness in working-age adults. Nephropathy: Kidney damage → chronic kidney disease → dialysis. Affects 20-40% of diabetics. Neuropathy: Nerve damage → numbness, pain, weakness. Peripheral (feet/hands) and autonomic (heart, digestion). MACROVASCULAR (large blood vessels): Cardiovascular disease: 2-4× higher risk of heart attack and stroke. Peripheral artery disease: Poor circulation → foot ulcers, amputation. PREVENTION: HbA1c <7.0% reduces microvascular risk by 25-75%. Blood pressure control (<130/80 mmHg). Lipid management (LDL <100 mg/dL). Annual eye exam, foot exam, kidney function tests. No smoking. GOOD NEWS: With modern management (CGM, insulin pumps, medications), many complications are preventable."
    }
]

print(f"Medical Q&A Dataset: {len(medical_qa_dataset)} examples")
print("\nSample entry:")
print(f"Prompt: {medical_qa_dataset[0]['prompt'][:100]}...")
print(f"Response: {medical_qa_dataset[0]['response'][:150]}...")

# Dataset statistics
import tiktoken
enc = tiktoken.get_encoding("cl100k_base")

prompt_lengths = [len(enc.encode(ex['prompt'])) for ex in medical_qa_dataset]
response_lengths = [len(enc.encode(ex['response'])) for ex in medical_qa_dataset]

print(f"\nDataset Statistics:")
print(f"  Avg prompt length:   {sum(prompt_lengths)/len(prompt_lengths):.0f} tokens")
print(f"  Avg response length: {sum(response_lengths)/len(response_lengths):.0f} tokens")
print(f"  Total tokens:        {sum(prompt_lengths) + sum(response_lengths):,}")

### 3.2 Format with Chat Template

In [ ]:
def format_with_chat_template(example: dict, system_prompt: str = None) -> str:
    """
    Format a Q&A pair with a chat template.
    Uses ChatML format (compatible with most open models).
    """
    if system_prompt is None:
        system_prompt = (
            "You are the GAS Advisor Agent, an AI assistant specialized in Type 1 and Type 2 "
            "diabetes management. You provide evidence-based, safe medical guidance. "
            "Always recommend consulting a healthcare provider for treatment decisions."
        )
    
    formatted = (
        f"<|im_start|>system\n{system_prompt}<|im_end|>\n"
        f"<|im_start|>user\n{example['prompt']}<|im_end|>\n"
        f"<|im_start|>assistant\n{example['response']}<|im_end|>"
    )
    return formatted


# Format all examples
formatted_dataset = [format_with_chat_template(ex) for ex in medical_qa_dataset]

print("Formatted example (first 500 chars):")
print("=" * 60)
print(formatted_dataset[0][:500])
print("...")
print("=" * 60)

# Show token counts with template
formatted_lengths = [len(enc.encode(ex)) for ex in formatted_dataset]
print(f"\nWith chat template:")
print(f"  Avg length: {sum(formatted_lengths)/len(formatted_lengths):.0f} tokens")
print(f"  Max length: {max(formatted_lengths)} tokens")
print(f"  Min length: {min(formatted_lengths)} tokens")

### 3.3 SFT Training with TRL

In [ ]:
if not TEST_MODE:
    import torch
    from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
    from trl import SFTTrainer, SFTConfig
    from datasets import Dataset
    
    print("Setting up SFT training...")
    
    # Use GPT-2 (small, fast for demo)
    model_name = "gpt2"
    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenizer.pad_token = tokenizer.eos_token
    
    model = AutoModelForCausalLM.from_pretrained(model_name)
    
    # Create HuggingFace dataset
    hf_dataset = Dataset.from_dict({"text": formatted_dataset})
    
    # Split train/eval
    split = hf_dataset.train_test_split(test_size=0.2, seed=42)
    train_dataset = split["train"]
    eval_dataset = split["test"]
    
    print(f"Train: {len(train_dataset)} examples")
    print(f"Eval:  {len(eval_dataset)} examples")
    
    # SFT Config
    sft_config = SFTConfig(
        output_dir="./gas_sft_model",
        num_train_epochs=3,
        per_device_train_batch_size=2,
        per_device_eval_batch_size=2,
        learning_rate=2e-5,
        warmup_steps=10,
        logging_steps=5,
        eval_strategy="epoch",
        save_strategy="epoch",
        max_length=512,
        report_to="none",
    )
    
    trainer = SFTTrainer(
        model=model,
        args=sft_config,
        train_dataset=train_dataset,
        eval_dataset=eval_dataset,
        processing_class=tokenizer,
    )
    
    print("Starting SFT training...")
    trainer.train()
    print("Training complete!")
    
    # Save training history
    training_history = trainer.state.log_history

else:
    print("TEST_MODE: Skipping actual SFT training")
    print("In production, this would fine-tune GPT-2 (or a larger model) on the medical Q&A dataset")
    
    # Mock training history
    training_history = [
        {'loss': 3.8, 'epoch': 0.5, 'step': 4},
        {'loss': 3.2, 'epoch': 1.0, 'step': 8, 'eval_loss': 3.5},
        {'loss': 2.9, 'epoch': 1.5, 'step': 12},
        {'loss': 2.6, 'epoch': 2.0, 'step': 16, 'eval_loss': 3.1},
        {'loss': 2.4, 'epoch': 2.5, 'step': 20},
        {'loss': 2.2, 'epoch': 3.0, 'step': 24, 'eval_loss': 2.9},
    ]
    print("\nMock training results:")
    for entry in training_history:
        if 'eval_loss' in entry:
            print(f"  Epoch {entry['epoch']:.0f}: train_loss={entry['loss']:.2f}, eval_loss={entry['eval_loss']:.2f}")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Plot training curves
train_losses = [(e['step'], e['loss']) for e in training_history if 'loss' in e and 'eval_loss' not in e]
eval_losses = [(e['step'], e['eval_loss']) for e in training_history if 'eval_loss' in e]

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Loss curves
ax1 = axes[0]
if train_losses:
    steps_t, losses_t = zip(*train_losses)
    ax1.plot(steps_t, losses_t, 'b-o', label='Train Loss', linewidth=2)
if eval_losses:
    steps_e, losses_e = zip(*eval_losses)
    ax1.plot(steps_e, losses_e, 'r-s', label='Eval Loss', linewidth=2)
ax1.set_xlabel('Training Step')
ax1.set_ylabel('Cross-Entropy Loss')
ax1.set_title('SFT Training Loss\n(Medical Q&A Dataset)', fontweight='bold')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Dataset composition
ax2 = axes[1]
categories = [
    'Glucose\nManagement', 'Insulin\nDosing', 'Hypoglycemia\nTreatment', 
    'Complications', 'Device\nManagement', 'Sick Day\nRules'
]
counts = [5, 4, 3, 3, 3, 2]
colors = ['#3498db', '#e74c3c', '#2ecc71', '#f39c12', '#9b59b6', '#1abc9c']
ax2.pie(counts, labels=categories, colors=colors, autopct='%1.0f%%', startangle=90)
ax2.set_title('Medical Q&A Dataset\nTopic Distribution', fontweight='bold')

plt.tight_layout()
plt.savefig('sft_training.png', dpi=150, bbox_inches='tight')
plt.show()

### 3.4 Evaluation: Base vs Fine-tuned

In [ ]:
# Held-out test questions
test_questions = [
    "Patient glucose is 350 mg/dL with large ketones. What should they do?",
    "What is the 15-15 rule for hypoglycemia treatment?",
    "How does exercise affect insulin sensitivity?",
]

if not TEST_MODE:
    # Generate responses from base and fine-tuned models
    print("Generating responses from base model vs fine-tuned model...")
    
    base_model = AutoModelForCausalLM.from_pretrained("gpt2")
    base_tokenizer = AutoTokenizer.from_pretrained("gpt2")
    base_tokenizer.pad_token = base_tokenizer.eos_token
    
    for q in test_questions[:2]:
        print(f"\nQuestion: {q}")
        
        # Base model
        inputs = base_tokenizer(q, return_tensors="pt", max_length=100, truncation=True)
        with torch.no_grad():
            out = base_model.generate(**inputs, max_new_tokens=50, do_sample=False, pad_token_id=base_tokenizer.eos_token_id)
        base_response = base_tokenizer.decode(out[0][inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        print(f"Base model: {base_response[:200]}")
        
        # Fine-tuned model (from trainer)
        ft_inputs = tokenizer(format_with_chat_template({'prompt': q, 'response': ''}), 
                              return_tensors="pt", max_length=200, truncation=True)
        with torch.no_grad():
            ft_out = trainer.model.generate(**ft_inputs, max_new_tokens=100, do_sample=False, 
                                            pad_token_id=tokenizer.eos_token_id)
        ft_response = tokenizer.decode(ft_out[0][ft_inputs['input_ids'].shape[1]:], skip_special_tokens=True)
        print(f"Fine-tuned: {ft_response[:200]}")

else:
    print("TEST_MODE: Showing expected behavior comparison")
    print("\n" + "=" * 70)
    print("COMPARISON: Base GPT-2 vs SFT Fine-tuned Model")
    print("=" * 70)
    
    comparisons = [
        {
            "question": "Patient glucose is 350 mg/dL with large ketones. What should they do?",
            "base": "The patient should eat more carbohydrates and rest. Glucose levels naturally fluctuate...",
            "finetuned": "Glucose 350 mg/dL + large ketones = HIGH RISK for DKA. IMMEDIATE ACTION: (1) Call healthcare provider or go to ER. (2) Do NOT exercise. (3) Administer correction insulin if advised. (4) Drink water aggressively. (5) Recheck every 2 hours..."
        },
        {
            "question": "What is the 15-15 rule for hypoglycemia?",
            "base": "The 15-15 rule is a mathematical concept used in various contexts...",
            "finetuned": "The 15-15 Rule: (1) Consume 15g fast-acting carbohydrates (glucose tablets, juice). (2) Wait 15 minutes. (3) Recheck glucose. (4) If still <70 mg/dL, repeat. This is the standard treatment for mild-moderate hypoglycemia."
        }
    ]
    
    for comp in comparisons:
        print(f"\nQ: {comp['question']}")
        print(f"\n❌ Base GPT-2: {comp['base']}")
        print(f"\n✅ SFT Model:  {comp['finetuned']}")
        print("-" * 70)

print("\n💡 Key insight: SFT dramatically improves domain-specific responses")
print("   But the base model's hallucinations show why we need alignment (DPO/PPO) next!")

---
## 🏥 4. GAS Connection

The GAS Advisor Agent starts as an SFT model trained on medical Q&A. This notebook creates the training data and fine-tuning pipeline.

**GAS SFT Pipeline:**

```
Step 1: Collect medical Q&A pairs (this notebook)
        ↓
Step 2: Format with chat template
        ↓  
Step 3: SFT training (TRL SFTTrainer)
        ↓
Step 4: Evaluate on held-out questions
        ↓
Step 5: Deploy as GAS Advisor Agent base model
        ↓
Step 6: Further align with DPO/PPO (Chapters 5-7)
```

**Why 20 examples is enough for a demo (but not production):**
- Production GAS would need 1,000-10,000 high-quality examples
- Examples should be reviewed by certified diabetes educators (CDEs)
- Include edge cases: pediatric patients, pregnancy, renal failure
- Balance topics: don't over-represent common cases

---
## ✅ 5. Checkpoint

### What We Built

| Component | Description |
|-----------|-------------|
| Medical Q&A dataset | 20 high-quality (prompt, response) pairs |
| `format_with_chat_template()` | ChatML formatter for training |
| SFT training pipeline | TRL SFTTrainer configuration |
| Base vs fine-tuned comparison | Qualitative evaluation |

### Key Takeaways

1. **SFT teaches format and domain knowledge** — the model learns to respond like a medical advisor
2. **Chat templates are critical** — they tell the model where the instruction ends and response begins
3. **SFT alone is insufficient** — the model can still give unsafe advice (just in the right format)
4. **Data quality > quantity** — 20 expert-reviewed examples beat 1000 noisy ones

### What Comes Next

**Chapter 5: PPO** — We'll add RL-based alignment to make the SFT model safer. PPO trains the model to maximize a reward signal (safety score) while staying close to the SFT baseline.

```
Ch4 ✅ (SFT) → Ch5 (PPO) → Ch6 (DPO) → Ch7 (GRPO)
```